In [77]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# kagglehub.dataset_download('<owner>/<dataset-slug>')

pd_data = pd.read_csv("/kaggle/input/datasets/shyamnadhs/heart-disease-prediction-dataset/disease_prediction.csv")

/kaggle/input/datasets/shyamnadhs/heart-disease-prediction-dataset/disease_prediction.csv


In [78]:
print(pd_data.head(5))

   patient_id  age  gender  glucose_mg_dl  cholesterol_mg_dl  systolic_bp  \
0           1   32    Male            101                235          152   
1           2   31    Male            124                191          134   
2           3   45    Male             57                141          114   
3           4   75  Female             69                268          120   
4           5   53    Male            107                163          131   

   diastolic_bp   bmi  heart_rate smoking alcohol_consumption  \
0            79  28.5          73      No                 Yes   
1            77  33.9          71      No                 Yes   
2            71  27.2          79     Yes                 Yes   
3            82  21.5          61     Yes                 Yes   
4            75  23.3          73     Yes                  No   

  physical_activity family_history disease  
0               Low            Yes     Yes  
1               Low            Yes     Yes  
2          

In [79]:
# lets get the categories of the physical activity 
physical = set(pd_data['physical_activity'])

print(physical)

gender = set(pd_data['gender'])

print(gender)

{'Low', 'High', 'Medium'}
{'Female', 'Male'}


In [17]:
#now lets transform  the male and the female
pd_data['gender']= pd_data['gender'].map({
    "Male" : 0,
    "Female" : 1
})

pd_data['smoking'] = pd_data['smoking'].map({
    "No" : 0,
    "Yes" : 1
})

pd_data['alcohol_consumption'] = pd_data['alcohol_consumption'].map({
    "No" : 0,
    "Yes": 1
})

pd_data['physical_activity'] = pd_data['physical_activity'].map({
    "High" : 0,
    "Medium" : 1,
    "Low" : 2
})

pd_data['family_history'] = pd_data['family_history'].map({
    "No" : 0,
    "Yes": 1
})

pd_data['disease'] = pd_data['disease'].map({
    "No" : 0,
    "Yes": 1
})

print(pd_data.head())

   patient_id  age  gender  glucose_mg_dl  cholesterol_mg_dl  systolic_bp  \
0           1   32       0            101                235          152   
1           2   31       0            124                191          134   
2           3   45       0             57                141          114   
3           4   75       1             69                268          120   
4           5   53       0            107                163          131   

   diastolic_bp   bmi  heart_rate  smoking  alcohol_consumption  \
0            79  28.5          73        0                    1   
1            77  33.9          71        0                    1   
2            71  27.2          79        1                    1   
3            82  21.5          61        1                    1   
4            75  23.3          73        1                    0   

   physical_activity  family_history  disease  
0                  2               1        1  
1                  2               1  

In [19]:
print(pd_data.info()) 
#this means there is no null values in our dataset 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   patient_id           1000 non-null   int64  
 1   age                  1000 non-null   int64  
 2   gender               1000 non-null   int64  
 3   glucose_mg_dl        1000 non-null   int64  
 4   cholesterol_mg_dl    1000 non-null   int64  
 5   systolic_bp          1000 non-null   int64  
 6   diastolic_bp         1000 non-null   int64  
 7   bmi                  1000 non-null   float64
 8   heart_rate           1000 non-null   int64  
 9   smoking              1000 non-null   int64  
 10  alcohol_consumption  1000 non-null   int64  
 11  physical_activity    1000 non-null   int64  
 12  family_history       1000 non-null   int64  
 13  disease              1000 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 109.5 KB
None


In [48]:
import numpy as np
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return np.where(z >= 0,
                    1 / (1 + np.exp(-z)),
                    np.exp(z) / (1 + np.exp(z)))


In [63]:
# now lets see the code for the logistic regression 
class Logistic_regression():

    def __init__(self , lr = 0.1 , epochs = 1000):
        self.lr = lr
        self.epochs = epochs
        self.weights = None
        self.bias = None

   #now lets write the function to fit the data to the model 
    def fit(self, X , y):
        #lets unpack the shape of X 
        m ,n = X.shape

        #after unpacking the number of value of m would be 1 and n would be number of features 
        self.weights = np.zeros(n)
        self.bias = 0 

        #now lets get the predictions of the model , and the prediction 
        for epoch in range(self.epochs):
            # now lets get the prediction of the model 
            y_hat = sigmoid(np.dot(X , self.weights) + self.bias) #this is the prediction of the model 
            # here we will print the error 
            #now lets get the  gradients of the  model 
            dw = (1/m)*np.dot(X.T,(y_hat-y))
            db = (1/m)* np.sum(y_hat-y)

             #now lets update the parameter
            self.weights -= self.lr*dw
            self.bias  -= self.lr*db
    #now lets predict the output of the model 
    def predict_proba(self,X):
        return sigmoid(np.dot(X , self.weights) + self.bias)

    def predict_label(self , X ,threshold = 0.5):
        predict_probab = self.predict_proba(X)
        return (predict_probab>=threshold).astype(int) # from here we get the label for the prediction
        
            

In [23]:
#now lets prepare the data for the training  part 
print(len(pd_data))
y = pd_data['disease']
X = pd_data.drop('disease' , axis = 1)
print(X)

1000
     patient_id  age  gender  glucose_mg_dl  cholesterol_mg_dl  systolic_bp  \
0             1   32       0            101                235          152   
1             2   31       0            124                191          134   
2             3   45       0             57                141          114   
3             4   75       1             69                268          120   
4             5   53       0            107                163          131   
..          ...  ...     ...            ...                ...          ...   
995         996   24       0            105                237           86   
996         997   40       0            120                219          133   
997         998   44       1            114                273          114   
998         999   31       0             95                231          130   
999        1000   83       0            109                256          103   

     diastolic_bp   bmi  heart_rate  smoking  

In [44]:
#now we have created our  model and the update rule as well ,now lets first seperate the data into the 
#train test and split and then  fit the model

from sklearn.model_selection import train_test_split
X_train  , y_train = X[:800] , y[:800] 
X_test , y_test = X[800:] , y[800:]
# now we have the train and the test set
print(len(X_train))

800


In [65]:
#now lets create the model 
classification_model = Logistic_regression(lr = 0.01 , epochs = 100000)

classification_model.fit(X_train , y_train)

#now lets get the predictions of the model 
preds = classification_model.predict_label(X_test)

#now lets get the accuracy of the model 
print(f"Accuracy  is {np.mean(y_test == preds)}")

Accuracy  is 0.575


In [72]:
!pip install pickle

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle


In [74]:
#now lets save the model 
import pickle as pkl
with open("logistic_model.pkl", "wb") as file:
    pkl.dump(classification_model, file)

In [75]:
!ls /kaggle/working

logistic_model.pkl
